In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import util

pd.set_option('display.float_format', '{:,.1%}'.format)

In [2]:
# add trip type to trip tables
def prep_trip_df(trip):
    """Mode shares will represent shares across Drive Alone, Shared Ride, 
       public transit, and walk and bike trips only. 
    """
    df_trip = trip.copy()
    df_trip.rename(columns={'mode':'Mode'}, inplace=True)
    df_trip['Trip Type'] = 'Non-Work'
    df_trip.loc[df_trip['dpurp'] == 'Work', 'Trip Type'] = 'Work'

    # add all tripa
    df = df_trip.copy()
    df['Trip Type'] = 'All Trips'
    df_trip = pd.concat([df_trip.sort_values('Trip Type'), df], ignore_index=True)

    # Exclude TNC, School Bus, and Other modes
    df_trip = df_trip[~df_trip['Mode'].isin(['TNC', 'School Bus', 'Other'])]

    # Aggregate modes
    df_trip['Mode'] = df_trip['Mode'].replace({
        'SOV': 'Drove Alone',
        'HOV2': 'Shared Ride',
        'HOV3+': 'Shared Ride',
        'Transit': 'Transit',
        'Walk': 'Walk',
        'Bike': 'Bike',
    })

    return df_trip

In [3]:
person = pd.read_csv(util.output_path / 'agg/dash/person_geog.csv')

# counties
trip_county = prep_trip_df(pd.read_csv(util.output_path / 'agg/dash/mode_share_county.csv'))
# remove "Outside Region"
trip_county = trip_county[trip_county['hh_county'] != 'Outside Region']
# regional geographies
trip_rg = prep_trip_df(pd.read_csv(util.output_path / 'agg/dash/mode_share_rg.csv'))
# centers
trip_rgc = pd.read_csv(util.output_path / 'agg/dash/mode_share_rgc.csv')
# add all RGCs
df = trip_rgc[trip_rgc['hh_rgc'] != 'Not in RGC'].groupby(['dpurp','mode']).sum()[['trexpfac']].reset_index()
df['hh_rgc'] = 'All RGCs'
trip_rgc = pd.concat([df, trip_rgc])
trip_rgc = prep_trip_df(trip_rgc)

In [4]:
# equity geographies
equity_geogs = util.summary_config['hh_equity_geogs']
trip_equity_geog = pd.DataFrame()
for geog in equity_geogs:
    df = prep_trip_df(pd.read_csv(util.output_path/ ('agg/dash/mode_share_'+geog+'.csv')))
    df['equity_geog'] = df[geog].map({
                                0: 'Below Regional Average', 
                                1: 'Above Regional Average', 
                                2: 'Higher Share of Equity Population'})
    
    trip_equity_geog = pd.concat([trip_equity_geog, df[['dpurp', 'Mode', 'equity_geog', 'trexpfac', 'Trip Type']]])


## regional mode share

In [5]:
df1 = trip_county.groupby(['Trip Type','Mode'],as_index=False)['trexpfac'].sum()
df2 = trip_county.groupby(['Trip Type'],as_index=False)['trexpfac'].sum()
df2.rename(columns={'trexpfac':'total_trips'}, inplace=True)

df = df1.merge(df2, on='Trip Type')
df['mode_share'] = df['trexpfac'] / df['total_trips']

df.pivot_table(columns='Mode', index='Trip Type', values='mode_share')

Mode,Bike,Drove Alone,Shared Ride,Transit,Walk
Trip Type,,,,,
All Trips,1.5%,45.7%,35.8%,2.2%,14.7%
Non-Work,1.5%,41.6%,39.1%,2.2%,15.6%
Work,1.4%,72.0%,15.2%,2.4%,9.0%


## trip mode share by home location

In [6]:
def calc_mode_share(df_trip, geog):
    
    # num trips by mode
    df1 = df_trip.groupby([geog, 'Trip Type', 'Mode'],as_index=False)['trexpfac'].sum()
    # num trips
    df2 = df_trip.groupby([geog, 'Trip Type'],as_index=False)['trexpfac'].sum()
    df2.rename(columns={'trexpfac':'total_trips'}, inplace=True)

    df = df1.merge(df2, on=[geog, 'Trip Type'])
    # mode share
    df['mode_share'] = df['trexpfac'] / df['total_trips']

    # mode share by trip type
    df_all = df.loc[df['Trip Type']=="All Trips"].pivot_table(columns='Mode', index=geog, values='mode_share')
    df_work = df.loc[df['Trip Type']=="Work"].pivot_table(columns='Mode', index=geog, values='mode_share')
    df_non_work = df.loc[df['Trip Type']=="Non-Work"].pivot_table(columns='Mode', index=geog, values='mode_share')
    # num trips by trip type
    trip_all = df.loc[df['Trip Type']=="All Trips"].pivot_table(columns='Mode', index=geog, values='trexpfac')
    trip_work = df.loc[df['Trip Type']=="Work"].pivot_table(columns='Mode', index=geog, values='trexpfac')
    trip_non_work = df.loc[df['Trip Type']=="Non-Work"].pivot_table(columns='Mode', index=geog, values='trexpfac')

    return df_all, df_work, df_non_work, trip_all, trip_work, trip_non_work


df_all, df_work, df_non_work, trip_all, trip_work, trip_non_work = calc_mode_share(trip_county,'hh_county')

In [7]:
df_all

Mode,Bike,Drove Alone,Shared Ride,Transit,Walk
hh_county,,,,,
King,1.7%,44.6%,33.7%,3.3%,16.7%
Kitsap,1.2%,46.0%,36.7%,1.7%,14.4%
Pierce,1.4%,46.2%,39.2%,0.9%,12.3%
Snohomish,1.3%,48.3%,38.0%,0.8%,11.5%


In [8]:
trip_all.style.format("{:0,.0f}")

Mode,Bike,Drove Alone,Shared Ride,Transit,Walk
hh_county,,,,,
King,"147,939","3,896,512","2,947,746","286,397","1,455,928"
Kitsap,"12,361","458,511","366,360","16,831","143,500"
Pierce,"45,812","1,493,967","1,267,533","28,427","399,488"
Snohomish,"40,002","1,505,961","1,184,072","26,272","359,685"


In [9]:
df_non_work

Mode,Bike,Drove Alone,Shared Ride,Transit,Walk
hh_county,,,,,
King,1.7%,40.4%,37.0%,3.3%,17.5%
Kitsap,1.3%,42.5%,39.7%,1.2%,15.3%
Pierce,1.5%,42.1%,42.4%,0.8%,13.3%
Snohomish,1.3%,43.9%,41.4%,0.9%,12.6%


In [10]:
trip_non_work.style.format("{:0,.0f}")

Mode,Bike,Drove Alone,Shared Ride,Transit,Walk
hh_county,,,,,
King,"125,775","3,016,787","2,761,884","248,091","1,306,946"
Kitsap,"11,220","373,741","348,600","10,875","134,623"
Pierce,"41,929","1,196,688","1,204,700","22,212","378,663"
Snohomish,"36,115","1,183,734","1,116,894","23,408","339,226"


In [11]:
df_work

Mode,Bike,Drove Alone,Shared Ride,Transit,Walk
hh_county,,,,,
King,1.7%,69.0%,14.6%,3.0%,11.7%
Kitsap,1.0%,71.5%,15.0%,5.0%,7.5%
Pierce,1.0%,76.0%,16.1%,1.6%,5.3%
Snohomish,0.9%,77.3%,16.1%,0.7%,4.9%


In [12]:
trip_work.style.format("{:0,.0f}")

Mode,Bike,Drove Alone,Shared Ride,Transit,Walk
hh_county,,,,,
King,"22,164","879,725","185,862","38,306","148,982"
Kitsap,"1,141","84,770","17,760","5,956","8,877"
Pierce,"3,883","297,279","62,833","6,215","20,825"
Snohomish,"3,887","322,227","67,178","2,864","20,459"


In [13]:
df_all, df_work, df_non_work, trip_all, trip_work, trip_non_work  = calc_mode_share(trip_rgc, 'hh_rgc')

df_all

Mode,Bike,Drove Alone,Shared Ride,Transit,Walk
hh_rgc,,,,,
All RGCs,3.2%,30.0%,16.3%,6.4%,44.1%
Auburn,2.1%,38.5%,28.8%,3.4%,27.2%
Bellevue,2.6%,33.1%,17.0%,3.6%,43.7%
Bothell Canyon Park,1.2%,52.4%,32.9%,0.9%,12.6%
Bremerton,2.7%,35.6%,22.3%,2.5%,36.8%
Burien,1.5%,45.0%,27.3%,3.3%,22.8%
Everett,2.0%,39.2%,20.9%,2.2%,35.7%
Federal Way,2.2%,37.3%,34.6%,2.3%,23.7%
Greater Downtown Kirkland,1.8%,48.2%,26.7%,3.8%,19.5%


In [14]:
trip_all.style.format("{:0,.0f}")

Mode,Bike,Drove Alone,Shared Ride,Transit,Walk
hh_rgc,,,,,
All RGCs,"35,453","333,224","181,239","71,179","489,376"
Auburn,178,"3,273","2,444",286,"2,315"
Bellevue,"1,722","22,132","11,370","2,415","29,197"
Bothell Canyon Park,30,"1,341",840,23,323
Bremerton,341,"4,529","2,837",324,"4,677"
Burien,226,"6,754","4,098",496,"3,424"
Everett,530,"10,336","5,516",582,"9,434"
Federal Way,49,842,781,51,534
Greater Downtown Kirkland,590,"15,416","8,554","1,208","6,245"


In [15]:
df_non_work

Mode,Bike,Drove Alone,Shared Ride,Transit,Walk
hh_rgc,,,,,
All RGCs,3.1%,27.0%,18.1%,6.8%,45.0%
Auburn,2.1%,35.1%,30.6%,2.8%,29.3%
Bellevue,2.5%,29.7%,18.7%,4.0%,45.2%
Bothell Canyon Park,1.1%,48.0%,36.2%,0.8%,13.9%
Bremerton,2.5%,32.3%,24.0%,1.9%,39.3%
Burien,1.4%,39.9%,29.4%,3.6%,25.7%
Everett,2.0%,34.8%,22.5%,2.3%,38.4%
Federal Way,2.2%,32.7%,37.5%,2.4%,25.2%
Greater Downtown Kirkland,1.8%,43.5%,29.3%,3.9%,21.5%


In [16]:
trip_non_work.style.format("{:0,.0f}")

Mode,Bike,Drove Alone,Shared Ride,Transit,Walk
hh_rgc,,,,,
All RGCs,"28,598","248,541","166,184","62,454","413,176"
Auburn,159,"2,635","2,301",214,"2,205"
Bellevue,"1,373","16,497","10,422","2,206","25,139"
Bothell Canyon Park,24,"1,037",781,18,300
Bremerton,276,"3,564","2,652",205,"4,337"
Burien,179,"5,086","3,743",454,"3,270"
Everett,450,"7,937","5,128",534,"8,740"
Federal Way,43,654,749,48,503
Greater Downtown Kirkland,500,"11,913","8,009","1,067","5,876"


In [17]:
df_work

Mode,Bike,Drove Alone,Shared Ride,Transit,Walk
hh_rgc,,,,,
All RGCs,3.6%,44.2%,7.9%,4.6%,39.8%
Auburn,1.9%,65.0%,14.6%,7.3%,11.2%
Bellevue,3.1%,50.3%,8.5%,1.9%,36.2%
Bothell Canyon Park,1.5%,76.6%,14.9%,1.3%,5.8%
Bremerton,3.9%,57.6%,11.1%,7.1%,20.3%
Burien,2.1%,73.6%,15.7%,1.9%,6.8%
Everett,2.2%,66.5%,10.8%,1.3%,19.2%
Federal Way,2.3%,72.3%,12.3%,1.2%,11.9%
Greater Downtown Kirkland,1.9%,75.4%,11.7%,3.0%,7.9%


In [18]:
trip_work.style.format("{:0,.0f}")

Mode,Bike,Drove Alone,Shared Ride,Transit,Walk
hh_rgc,,,,,
All RGCs,"6,855","84,683","15,055","8,725","76,200"
Auburn,19,638,143,72,110
Bellevue,349,"5,635",948,209,"4,058"
Bothell Canyon Park,6,304,59,5,23
Bremerton,65,965,185,119,340
Burien,47,"1,668",355,42,154
Everett,80,"2,399",388,48,694
Federal Way,6,188,32,3,31
Greater Downtown Kirkland,90,"3,503",545,141,369


In [19]:
df_all, df_work, df_non_work, trip_all, trip_work, trip_non_work  = calc_mode_share(trip_rg, 'hh_rg_proposed')

df_all

Mode,Bike,Drove Alone,Shared Ride,Transit,Walk
hh_rg_proposed,,,,,
Cities and Towns,1.2%,46.6%,39.7%,0.6%,11.9%
Core Cities,1.4%,46.9%,37.5%,1.9%,12.3%
High Capacity Transit Communities,1.3%,47.5%,38.3%,1.4%,11.5%
Metropolitan Cities,2.1%,41.2%,28.8%,4.4%,23.4%
Rural Areas,0.9%,51.2%,41.3%,0.5%,6.1%
Urban Unincorporated Areas,1.2%,46.3%,42.5%,0.5%,9.5%


In [20]:
trip_all.style.format("{:0,.0f}")

Mode,Bike,Drove Alone,Shared Ride,Transit,Walk
hh_rg_proposed,,,,,
Cities and Towns,"16,521","649,884","554,863","8,673","165,968"
Core Cities,"52,790","1,737,408","1,390,587","70,571","457,010"
High Capacity Transit Communities,"46,165","1,664,168","1,341,872","50,461","403,069"
Metropolitan Cities,"105,930","2,031,702","1,419,928","215,077","1,155,675"
Rural Areas,"17,184","978,588","789,283","9,770","116,822"
Urban Unincorporated Areas,"7,524","293,212","269,190","3,375","60,057"


In [21]:
df_non_work

Mode,Bike,Drove Alone,Shared Ride,Transit,Walk
hh_rg_proposed,,,,,
Cities and Towns,1.3%,42.1%,43.1%,0.6%,12.9%
Core Cities,1.5%,42.5%,40.8%,1.9%,13.3%
High Capacity Transit Communities,1.4%,43.1%,41.6%,1.4%,12.5%
Metropolitan Cities,2.1%,37.2%,31.6%,4.5%,24.6%
Rural Areas,1.0%,47.4%,44.8%,0.4%,6.4%
Urban Unincorporated Areas,1.3%,42.0%,46.0%,0.5%,10.3%


In [22]:
trip_non_work.style.format("{:0,.0f}")

Mode,Bike,Drove Alone,Shared Ride,Transit,Walk
hh_rg_proposed,,,,,
Cities and Towns,"15,422","512,302","524,089","6,934","157,166"
Core Cities,"46,669","1,362,068","1,309,101","59,786","427,510"
High Capacity Transit Communities,"41,730","1,311,320","1,266,831","41,904","379,861"
Metropolitan Cities,"87,960","1,554,428","1,323,568","186,872","1,029,573"
Rural Areas,"16,187","797,228","752,168","6,572","108,148"
Urban Unincorporated Areas,"7,071","233,614","256,333","2,518","57,200"


In [23]:
df_work

Mode,Bike,Drove Alone,Shared Ride,Transit,Walk
hh_rg_proposed,,,,,
Cities and Towns,0.6%,76.4%,17.1%,1.0%,4.9%
Core Cities,1.2%,74.6%,16.2%,2.1%,5.9%
High Capacity Transit Communities,1.0%,76.0%,16.2%,1.8%,5.0%
Metropolitan Cities,2.4%,64.0%,12.9%,3.8%,16.9%
Rural Areas,0.4%,78.4%,16.0%,1.4%,3.7%
Urban Unincorporated Areas,0.6%,77.8%,16.8%,1.1%,3.7%


In [24]:
trip_work.style.format("{:0,.0f}")

Mode,Bike,Drove Alone,Shared Ride,Transit,Walk
hh_rg_proposed,,,,,
Cities and Towns,"1,099","137,582","30,774","1,739","8,802"
Core Cities,"6,121","375,340","81,486","10,785","29,500"
High Capacity Transit Communities,"4,435","352,848","75,041","8,557","23,208"
Metropolitan Cities,"17,970","477,274","96,360","28,205","126,102"
Rural Areas,997,"181,360","37,115","3,198","8,674"
Urban Unincorporated Areas,453,"59,598","12,857",857,"2,857"


In [25]:
equity_geogs = util.summary_config['hh_equity_geogs']

trip_equity_geog = pd.DataFrame()
df_all = pd.DataFrame()
df_work = pd.DataFrame()
df_non_work = pd.DataFrame()
trip_work = pd.DataFrame()
trip_non_work = pd.DataFrame()

for geog in equity_geogs:
    df = prep_trip_df(pd.read_csv(util.output_path/ ('agg/dash/mode_share_'+geog+'.csv')))
    df['equity_geog'] = df[geog].map({
                                0: 'Below Regional Average', 
                                1: 'Above Regional Average', 
                                2: 'Higher Share of Equity Population'})
    df['EFA'] = geog
    # trip_equity_geog = pd.concat([trip_equity_geog, df[['equity_geog_type', 'dpurp', 'Mode', 'equity_geog','trexpfac', 'Trip Type']]])

    _df_all, _df_work, _df_non_work, _trip_all, _trip_work, _trip_non_work = calc_mode_share(df, 'equity_geog')

    _df_all['EFA'] = geog
    _df_work['EFA'] = geog
    _df_non_work['EFA'] = geog
    _trip_all['EFA'] = geog
    _trip_work['EFA'] = geog
    _trip_non_work['EFA'] = geog


    df_all = pd.concat([df_all, _df_all])
    df_work = pd.concat([df_work, _df_work])
    df_non_work = pd.concat([df_non_work, _df_non_work])
    trip_all = pd.concat([trip_all, _trip_all])
    trip_work = pd.concat([trip_work, _trip_work])
    trip_non_work = pd.concat([trip_non_work, _trip_non_work])

col_dict = {'hh_efa_dis': 'Disability',
                                'hh_efa_pov200': 'Income',
                                'hh_efa_poc': 'People of Color',
                                'hh_efa_lep': 'LEP',
                                'hh_efa_older': 'Older Adult',
                                'hh_efa_youth': 'Youth'}
col_list = ['EFA','Drove Alone', 'Shared Ride', 'Transit','Walk', 'Bike']

df_all['EFA'] = df_all['EFA'].map(col_dict)
df_all = df_all[col_list]
df_work['EFA'] = df_work['EFA'].map(col_dict)
df_work = df_work[col_list]
df_non_work['EFA'] = df_non_work['EFA'].map(col_dict)
df_non_work = df_non_work[col_list]
trip_all['EFA'] = trip_all['EFA'].map(col_dict)
trip_all = trip_all[col_list]
trip_work['EFA'] = trip_work['EFA'].map(col_dict)
trip_work = trip_work[col_list]
trip_non_work['EFA'] = trip_non_work['EFA'].map(col_dict)
trip_non_work = trip_non_work[col_list]

In [26]:
df_all

Mode,EFA,Drove Alone,Shared Ride,Transit,Walk,Bike
equity_geog,,,,,,
Above Regional Average,Disability,45.7%,35.8%,2.2%,14.8%,1.5%
Below Regional Average,Disability,46.2%,36.3%,2.3%,13.7%,1.5%
Higher Share of Equity Population,Disability,43.8%,34.3%,2.2%,18.1%,1.6%
Above Regional Average,Older Adult,46.6%,36.5%,2.0%,13.6%,1.4%
Below Regional Average,Older Adult,44.8%,35.6%,2.4%,15.6%,1.7%
Higher Share of Equity Population,Older Adult,47.4%,35.3%,2.3%,13.8%,1.3%
Above Regional Average,LEP,46.0%,36.5%,2.1%,13.9%,1.5%
Below Regional Average,LEP,45.9%,35.2%,2.2%,15.2%,1.5%
Higher Share of Equity Population,LEP,44.9%,37.3%,2.7%,13.6%,1.6%


In [27]:
df_non_work

Mode,EFA,Drove Alone,Shared Ride,Transit,Walk,Bike
equity_geog,,,,,,
Above Regional Average,Disability,41.7%,38.9%,2.1%,15.7%,1.5%
Below Regional Average,Disability,41.9%,39.8%,2.2%,14.6%,1.5%
Higher Share of Equity Population,Disability,40.1%,37.1%,2.2%,19.0%,1.6%
Above Regional Average,Older Adult,42.6%,39.7%,2.0%,14.4%,1.4%
Below Regional Average,Older Adult,40.4%,39.0%,2.3%,16.6%,1.7%
Higher Share of Equity Population,Older Adult,43.8%,38.1%,2.2%,14.6%,1.3%
Above Regional Average,LEP,41.8%,39.9%,2.1%,14.7%,1.5%
Below Regional Average,LEP,41.8%,38.4%,2.1%,16.1%,1.6%
Higher Share of Equity Population,LEP,40.4%,40.6%,2.7%,14.7%,1.6%


In [28]:
df_work

Mode,EFA,Drove Alone,Shared Ride,Transit,Walk,Bike
equity_geog,,,,,,
Above Regional Average,Disability,72.1%,15.3%,2.4%,8.8%,1.4%
Below Regional Average,Disability,72.3%,15.2%,2.5%,8.6%,1.4%
Higher Share of Equity Population,Disability,69.8%,14.6%,2.4%,11.6%,1.6%
Above Regional Average,Older Adult,73.0%,15.4%,2.1%,8.2%,1.2%
Below Regional Average,Older Adult,71.1%,15.1%,2.4%,9.8%,1.6%
Higher Share of Equity Population,Older Adult,73.1%,14.9%,3.2%,7.7%,1.1%
Above Regional Average,LEP,72.3%,15.5%,1.9%,9.0%,1.3%
Below Regional Average,LEP,71.7%,14.7%,2.6%,9.6%,1.4%
Higher Share of Equity Population,LEP,72.6%,16.4%,2.3%,7.2%,1.5%


## Mode Share by Trip Destination

In [29]:
tour_rgc_dest = pd.read_csv(util.output_path / 'agg/dash/tour_rgc_dest.csv')
trip_rgc_dest = pd.read_csv(util.output_path / 'agg/dash/trip_rgc_dest.csv')

# order RGCs

# all individual centers
l = tour_rgc_dest['tour_d_rgc'].sort_values().unique().tolist()
l.remove('Not in RGC')
l.remove(np.nan)

# aggregated centers
geog_order = ["Region","In RGC","Not in RGC"]

# combine
geog_order.extend(l)

In [30]:
def get_rgc_location(df, dest_col):
    # regional data
    df_region = df.copy()
    df_region[dest_col] = "Region"

    # aggregated centers
    df_isrgc = df.copy()
    df_isrgc.loc[df_isrgc[dest_col]!="Not in RGC", dest_col] = "In RGC"


    df_rgc = pd.concat([
        df_region,
        df_isrgc,
    # 2025 notes:
    # 7 records in the data are missing both mode and and RGC values
    # remove these records
        df.loc[~df[dest_col].isna()]
        ])

    df_rgc[dest_col] = pd.Categorical(df_rgc[dest_col], ordered=True,
                    categories=geog_order)
    
    return df_rgc

df_tour = get_rgc_location(tour_rgc_dest, 'tour_d_rgc')
df_trip = get_rgc_location(trip_rgc_dest, 'trip_d_rgc')




In [31]:
def calc_mode_share_by_rgc(df_trip_geog, geog_col, mode_col, n_trip_col):

    df = df_trip_geog.pivot_table(columns=mode_col, index=geog_col, values=n_trip_col, aggfunc='sum', observed=True)

    df = df.apply(lambda x: x/ df_trip_geog.groupby(geog_col, observed=True)[n_trip_col].sum())

    return df

\[TRIP\] Mode Share for All Purposes by Regional Center (Destination Location)

In [32]:
calc_mode_share_by_rgc(df_trip, 'trip_d_rgc', 'mode', 'trexpfac')

mode,Bike,HOV2,HOV3+,SOV,School Bus,Transit,Walk
trip_d_rgc,,,,,,,
Region,1.5%,21.2%,14.1%,45.0%,1.6%,2.2%,14.4%
In RGC,2.0%,15.9%,8.8%,42.6%,0.5%,5.6%,24.8%
Not in RGC,1.4%,22.3%,15.1%,45.5%,1.8%,1.5%,12.4%
Auburn,1.1%,21.4%,12.2%,48.2%,0.4%,4.2%,12.4%
Bellevue,1.6%,15.0%,8.7%,44.0%,0.4%,2.6%,27.7%
Bothell Canyon Park,1.0%,22.1%,13.8%,55.0%,1.1%,0.4%,6.6%
Bremerton,1.6%,15.7%,8.6%,55.5%,0.1%,1.7%,16.8%
Burien,1.0%,23.1%,12.7%,46.3%,0.9%,2.4%,13.6%
Everett,1.2%,18.6%,11.0%,48.4%,0.6%,1.6%,18.7%


\[TRIP\] Mode Share to Work by Regional Center (Destination Location)

In [33]:
df_trip_work = df_trip[df_trip['dpurp'] == 'Work'].copy()
calc_mode_share_by_rgc(df_trip_work, 'trip_d_rgc', 'mode', 'trexpfac')

mode,Bike,HOV2,HOV3+,SOV,School Bus,Transit,Walk
trip_d_rgc,,,,,,,
Region,1.4%,9.3%,5.8%,72.0%,0.0%,2.4%,9.0%
In RGC,1.9%,8.5%,5.3%,62.1%,0.0%,5.3%,16.9%
Not in RGC,1.2%,9.8%,6.1%,77.1%,0.0%,1.0%,5.0%
Auburn,1.3%,10.0%,6.3%,73.7%,0.0%,1.5%,7.1%
Bellevue,1.5%,9.0%,6.0%,65.6%,NaN,1.8%,16.1%
Bothell Canyon Park,1.0%,9.7%,5.9%,79.7%,0.0%,0.2%,3.6%
Bremerton,1.2%,10.2%,6.2%,75.2%,NaN,1.3%,5.9%
Burien,1.2%,10.0%,6.3%,74.7%,0.0%,1.2%,6.6%
Everett,1.1%,9.0%,5.5%,73.3%,NaN,0.5%,10.6%


\[TRIP\] Mode Share for Non-work Purposes by Regional Center (Destination Location)

In [34]:
df_trip_non_work = df_trip[df_trip['dpurp'] != 'Work'].copy()
calc_mode_share_by_rgc(df_trip_non_work, 'trip_d_rgc', 'mode', 'trexpfac')

mode,Bike,HOV2,HOV3+,SOV,School Bus,Transit,Walk
trip_d_rgc,,,,,,,
Region,1.5%,23.1%,15.3%,40.8%,1.8%,2.2%,15.3%
In RGC,2.0%,18.7%,10.2%,34.9%,0.6%,5.7%,27.9%
Not in RGC,1.4%,23.8%,16.1%,41.7%,2.0%,1.6%,13.3%
Auburn,1.1%,23.4%,13.2%,43.6%,0.5%,4.7%,13.4%
Bellevue,1.7%,18.0%,10.1%,32.9%,0.6%,2.9%,33.7%
Bothell Canyon Park,1.0%,27.3%,17.1%,44.5%,1.6%,0.5%,7.9%
Bremerton,2.1%,21.0%,10.8%,36.2%,0.3%,2.1%,27.6%
Burien,1.0%,24.5%,13.4%,43.3%,1.0%,2.5%,14.3%
Everett,1.3%,21.3%,12.5%,41.4%,0.7%,1.9%,20.9%


## Mode Share by Tour Destination

\[TOUR\] Mode Share for All Purposes by Regional Center (Destination Location)

In [35]:
calc_mode_share_by_rgc(df_tour, 'tour_d_rgc', 'tmodetp', 'toexpfac')

tmodetp,Bike,HOV2,HOV3+,Park,SOV,School Bus,Transit,Walk
tour_d_rgc,,,,,,,,
Region,1.4%,22.1%,19.3%,0.2%,38.4%,2.5%,3.2%,13.0%
In RGC,1.6%,18.4%,14.1%,0.6%,40.0%,0.6%,7.9%,16.7%
Not in RGC,1.3%,23.1%,20.7%,0.1%,37.9%,3.1%,1.9%,11.9%
Auburn,0.8%,24.9%,20.0%,0.2%,42.5%,0.3%,2.5%,8.9%
Bellevue,1.3%,17.8%,14.2%,0.3%,44.2%,0.3%,3.3%,18.6%
Bothell Canyon Park,1.0%,23.9%,21.4%,0.1%,46.2%,2.0%,0.7%,4.7%
Bremerton,1.3%,21.0%,15.4%,0.1%,51.1%,0.1%,0.9%,10.0%
Burien,0.8%,24.2%,18.6%,0.1%,39.7%,1.6%,3.9%,11.2%
Everett,1.0%,22.0%,17.8%,0.1%,41.9%,1.0%,2.3%,13.9%


\[TOUR\] Mode Share to Work by Regional Center (Destination Location)

In [36]:
df_tour_work = df_tour[df_tour['pdpurp'] == 'Work'].copy()
calc_mode_share_by_rgc(df_tour_work, 'tour_d_rgc', 'tmodetp', 'toexpfac')

tmodetp,Bike,HOV2,HOV3+,Park,SOV,Transit,Walk
tour_d_rgc,,,,,,,
Region,1.4%,15.3%,12.5%,0.6%,62.8%,2.9%,4.5%
In RGC,1.9%,14.0%,11.3%,1.3%,55.7%,6.4%,9.4%
Not in RGC,1.1%,16.0%,13.1%,0.3%,66.6%,1.0%,1.9%
Auburn,1.1%,17.4%,13.7%,0.6%,63.3%,1.7%,2.2%
Bellevue,1.5%,15.5%,13.0%,0.5%,59.8%,2.1%,7.5%
Bothell Canyon Park,1.0%,15.7%,13.0%,0.2%,69.3%,0.1%,0.7%
Bremerton,1.0%,17.1%,13.5%,0.2%,65.1%,1.4%,1.7%
Burien,1.1%,16.8%,12.7%,0.3%,64.9%,1.3%,2.9%
Everett,1.2%,16.1%,12.9%,0.2%,64.2%,0.4%,5.0%


\[TOUR\] Mode Share for Non-work Purposes by Regional Center (Destination Location)

In [37]:
df_tour_non_work = df_tour[df_tour['pdpurp'] != 'Work'].copy()
calc_mode_share_by_rgc(df_tour_non_work, 'tour_d_rgc', 'tmodetp', 'toexpfac')

tmodetp,Bike,HOV2,HOV3+,SOV,School Bus,Transit,Walk
tour_d_rgc,,,,,,,
Region,1.3%,24.7%,21.9%,28.8%,3.5%,3.4%,16.3%
In RGC,1.3%,22.0%,16.4%,27.4%,1.1%,9.0%,22.6%
Not in RGC,1.3%,25.3%,23.1%,29.1%,4.0%,2.2%,15.0%
Auburn,0.7%,27.5%,22.2%,35.3%,0.3%,2.8%,11.3%
Bellevue,0.9%,21.1%,15.7%,22.6%,0.8%,5.0%,33.9%
Bothell Canyon Park,0.9%,29.6%,27.2%,30.2%,3.4%,1.2%,7.5%
Bremerton,1.8%,27.5%,18.4%,27.7%,0.3%,0.3%,24.1%
Burien,0.8%,25.5%,19.7%,35.0%,1.9%,4.4%,12.7%
Everett,0.9%,24.7%,19.9%,31.9%,1.4%,3.2%,17.9%
